# HƯỚNG DẪN HUGGING FACE TRANSFORMERS — TỪ CƠ BẢN ĐẾN ỨNG DỤNG CHO TIẾNG VIỆT

**Biên soạn và giảng dạy:** TS. Hoàng Văn Quý
**Học phần:** Deep Learning trong NLP · **Phiên bản:** 2.0 (Colab, cập nhật 09/2026)
**Tư liệu tham khảo khung bài:** tutorial Hugging Face Transformers của Stanford CS224N (tác giả gốc Ben Newman, cập nhật Winter 2026 bởi Minsik Oh) [39].

---

### Mục tiêu
Sau bài học, sinh viên có thể:
1. Hiểu và dùng đúng bộ ba **Tokenizer → Model → Hậu xử lý** của thư viện `transformers` [1], cùng `datasets` [2].
2. Nắm các chi tiết hay gây lỗi: fast tokenizer, `offset_mapping`, `word_ids()`, padding động, padding trái cho decoder, chat template.
3. Fine-tune bằng **vòng lặp PyTorch** và bằng **`Trainer`**, lưu/tải checkpoint đúng cách, fine-tune tiết kiệm bằng **LoRA/PEFT** [21, 22].
4. Áp dụng cho **tiếng Việt**: chuẩn hoá Unicode, tách từ cho PhoBERT, chọn tokenizer/mô hình phù hợp, fine-tune phân loại, căn nhãn NER, tóm tắt, embedding và LLM hội thoại tiếng Việt.

### Bản đồ bài học
| Phần | Nội dung | Thời lượng |
|---|---|---|
| 0 | Mẫu sử dụng chung · Tokenizer · Model | 40' |
| 1 | Fine-tuning: vòng lặp PyTorch, `Trainer`, callback, LoRA | 45' |
| 2 | Sinh văn bản, chat template, dataset tuỳ chỉnh, pipeline, MLM | 30' |
| 3 | **Ứng dụng cho tiếng Việt** | 60' |
| 4 | Bài tập, tài liệu tham khảo | — |

> **Quy ước:** mỗi mục mở đầu bằng khung **⏱ Ý chính** (đọc 30 giây). Số trong ngoặc vuông như **[4]** trỏ tới *Tài liệu tham khảo* ở cuối notebook.

### Vì sao cần Hugging Face?
Thư viện `transformers` [1] thống nhất hàng trăm kiến trúc (BERT [4], GPT-2 [7], T5 [9], BART [8], Llama, Qwen [37]…) dưới **một giao diện chung** (`AutoTokenizer`, `AutoModel…`, `pipeline`, `Trainer`), kèm Hub lưu trữ trọng số, dataset và model card. Theo cách phân loại project của CS224N [39], thư viện đặc biệt hữu ích cho: (1) áp dụng mô hình tiền huấn luyện vào bài toán mới, và (3) phân tích hành vi mô hình. Với (2) thiết kế kiến trúc mới, xem hướng dẫn *custom models* trong tài liệu [40].

In [ ]:
# ============================================================
# CÀI ĐẶT MÔI TRƯỜNG (chạy 1 lần; nếu Colab yêu cầu restart thì restart rồi chạy tiếp)
# ============================================================
!pip -q install -U transformers datasets evaluate accelerate peft sentencepiece \
    scikit-learn sentence-transformers underthesea pyvi

In [ ]:
from collections import defaultdict, Counter
import json, math, os, random, re, time, unicodedata

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from matplotlib import pyplot as plt
import transformers

SEED = 224
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers:", transformers.__version__, "| torch:", torch.__version__, "| device:", device)

def print_encoding(model_inputs, indent=4):
    """In gọn một BatchEncoding (dict các tensor/list)."""
    ind = " " * indent
    print("{")
    for k, v in model_inputs.items():
        print(ind + k + ":")
        print(ind * 2 + str(v))
    print("}")

# PHẦN 0 — MẪU SỬ DỤNG CHUNG CỦA HUGGING FACE

> **⏱ Ý chính.** Mọi tác vụ đều đi qua 3 bước: **tokenizer** biến chuỗi thành ID → **model** biến ID thành *logits* → **hậu xử lý** (softmax, argmax, ánh xạ nhãn). Tokenizer và model **phải đi cùng một checkpoint**.

Ví dụ dùng mô hình **SiEBERT** (`siebert/sentiment-roberta-large-english`) — RoBERTa-large [6] được tinh chỉnh trên 15 bộ dữ liệu cảm xúc tiếng Anh [10].

![full_nlp_pipeline](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/full_nlp_pipeline.svg)
*Nguồn hình: Hugging Face LLM Course, Chương 2 [40].*

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ckpt = "siebert/sentiment-roberta-large-english"
tokenizer = AutoTokenizer.from_pretrained(ckpt)
model = AutoModelForSequenceClassification.from_pretrained(ckpt).eval()

text = "I'm excited to learn about Hugging Face Transformers!"
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():                         # suy luận: không cần tính gradient
    outputs = model(**inputs)

probs = outputs.logits.softmax(dim=-1)[0]
pred = int(probs.argmax())
print("Đầu vào đã token hoá:"); print_encoding(inputs)
print("Logits:", outputs.logits)
# Lấy tên nhãn từ config thay vì tự gõ tay → không bao giờ bị đảo thứ tự nhãn
print(f"Dự đoán: {model.config.id2label[pred]} (p = {probs[pred]:.3f})")

## 0.1 Tokenizer

> **⏱ Ý chính.** Tokenizer = *chuẩn hoá → tách sơ bộ → thuật toán subword → thêm token đặc biệt → ánh xạ ID*. **Fast tokenizer** (viết bằng Rust) nhanh hơn và có thêm các hàm căn chỉnh (`offset_mapping`, `word_ids()`) — rất cần cho NER/QA.

### Ba họ thuật toán subword
| Thuật toán | Nguyên tắc gộp | Mô hình dùng | Dấu hiệu nhận biết |
|---|---|---|---|
| WordPiece [14] | Gộp cặp làm tăng likelihood nhiều nhất | BERT, DistilBERT | Tiền tố `##` cho mảnh không đứng đầu từ |
| BPE / byte-level BPE [15] | Gộp cặp xuất hiện nhiều nhất (trên byte) | GPT-2, RoBERTa, Llama 3, Qwen | `Ġ` đánh dấu khoảng trắng phía trước |
| SentencePiece (Unigram/BPE) [16] | Học trực tiếp trên chuỗi thô, coi khoảng trắng là ký tự | XLM-R, T5, ViT5, ViSoBERT | `▁` đánh dấu đầu từ |

### Slow vs Fast
Mỗi mô hình có lớp tokenizer riêng (ví dụ `DistilBertTokenizer`) hoặc dùng `AutoTokenizer` (mặc định chọn bản **fast** nếu có). Kiểm tra bằng thuộc tính `tokenizer.is_fast`. Lưu ý: một số tokenizer (ví dụ **PhoBERT**) chỉ có bản slow → không có `word_ids()`; xem cách xử lý ở Phần 3.

In [ ]:
from transformers import DistilBertTokenizer, DistilBertTokenizerFast

name = "distilbert/distilbert-base-cased"
# name = "user/name" khi tải từ Hub; name = đường_dẫn_cục_bộ khi đã save_pretrained()

for cls in [DistilBertTokenizer, DistilBertTokenizerFast, AutoTokenizer]:
    t = cls.from_pretrained(name)
    print(f"{cls.__name__:25s} → {type(t).__name__:25s} is_fast={t.is_fast}")

tokenizer = AutoTokenizer.from_pretrained(name)   # dùng tiếp trong các cell sau

In [ ]:
# Gọi tokenizer như một hàm
input_str = "Hugging Face Transformers is great!"
enc = tokenizer(input_str)
print("Kết quả tokenizer(input_str):"); print_encoding(enc)
print("Hai cách truy cập giống nhau:", enc.input_ids == enc["input_ids"])

In [ ]:
# Tokenize "bóc tách" từng bước — để thấy tokenizer làm gì bên trong
input_tokens = tokenizer.tokenize(input_str)
input_ids = tokenizer.convert_tokens_to_ids(input_tokens)
with_special = [tokenizer.cls_token_id] + input_ids + [tokenizer.sep_token_id]

print("chuỗi gốc            :", input_str)
print("tokenize             :", input_tokens)
print("convert_tokens_to_ids:", input_ids)
print("thêm token đặc biệt  :", with_special)
print("decode               :", tokenizer.decode(with_special))
# LƯU Ý: cách thủ công này KHÔNG tạo attention_mask, không padding/truncation.
# Trong thực tế luôn gọi tokenizer(...) trực tiếp.

In [ ]:
# Hàm căn chỉnh của FAST tokenizer (API công khai, không dùng tokenizer._tokenizer)
enc = tokenizer(input_str, return_offsets_mapping=True)
tokens = enc.tokens()
print(f"{'token':12s} {'offset':10s} {'word_id':8s}")
for tok_, off, wid in zip(tokens, enc["offset_mapping"], enc.word_ids()):
    print(f"{tok_:12s} {str(off):10s} {str(wid):8s}")

char_idx = 8
t_idx = enc.char_to_token(char_idx)
print(f"\nKý tự thứ {char_idx+1} là '{input_str[char_idx]}' → thuộc token #{t_idx} '{tokens[t_idx]}'")
print("Ứng dụng: offset_mapping dùng để trích span đáp án (QA), word_ids() dùng để căn nhãn NER.")

### Padding, truncation và văn bản dài

- **Padding động (dynamic padding)**: chỉ đệm tới độ dài câu dài nhất *trong batch* (`DataCollatorWithPadding`) — nhanh hơn nhiều so với `padding="max_length"` luôn đệm tới 512.
- **`attention_mask`** = 1 ở token thật, 0 ở token đệm → attention bỏ qua phần đệm.
- **Văn bản dài hơn giới hạn** (BERT 512, PhoBERT 256): dùng `truncation=True, max_length=..., stride=..., return_overflowing_tokens=True` để cắt thành nhiều cửa sổ chồng lấn — kỹ thuật chuẩn cho QA trên tài liệu dài [40].

In [ ]:
batch = tokenizer(
    ["Hugging Face Transformers is great!",
     "The quick brown fox jumps over the lazy dog. Then the dog got up and ran away because she didn't like foxes."],
    padding=True, truncation=True, return_tensors="pt")
print(f"pad_token = {tokenizer.pad_token!r}, id = {tokenizer.pad_token_id}")
print_encoding(batch)

print("\nbatch_decode (giữ token đặc biệt):", tokenizer.batch_decode(batch.input_ids))
print("batch_decode (bỏ token đặc biệt) :", tokenizer.batch_decode(batch.input_ids, skip_special_tokens=True))

# Văn bản dài: cắt thành cửa sổ trượt 16 token, chồng lấn 4 token
long_text = " ".join(["Transformers make transfer learning easy."] * 10)
win = tokenizer(long_text, max_length=16, truncation=True, stride=4, return_overflowing_tokens=True)
print(f"\nVăn bản dài → {len(win['input_ids'])} cửa sổ; cửa sổ 1: {tokenizer.decode(win['input_ids'][0])}")

## 0.2 Model

> **⏱ Ý chính.** Một checkpoint = **thân** (body, phần Transformer tiền huấn luyện) + **đầu** (head, lớp nhỏ theo tác vụ). Chọn lớp `AutoModelFor<Tác_vụ>` là chọn head. Head mới được **khởi tạo ngẫu nhiên** → phải fine-tune.

![transformer_and_head](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/transformer_and_head.svg)
*Nguồn hình: Hugging Face LLM Course, Chương 2 [40].*

| Lớp | Head | Đầu ra | Dùng cho |
|---|---|---|---|
| `AutoModel` | không có | `last_hidden_state` | Trích đặc trưng, embedding |
| `AutoModelForMaskedLM` | LM head | logits trên vocab ở mọi vị trí | MLM, tiếp tục pre-train |
| `AutoModelForCausalLM` | LM head | logits token kế tiếp | Sinh văn bản (GPT, Qwen) |
| `AutoModelForSeq2SeqLM` | Encoder–decoder + LM head | logits của decoder | Tóm tắt, dịch (T5, BART) |
| `AutoModelForSequenceClassification` | Linear trên biểu diễn câu | logits theo lớp | Phân loại câu/cặp câu |
| `AutoModelForTokenClassification` | Linear trên từng token | logits theo nhãn mỗi token | NER, POS |
| `AutoModelForQuestionAnswering` | 2 vector start/end | logits vị trí đầu/cuối | QA trích xuất |

Ba họ kiến trúc [3]: **encoder** (BERT [4], DistilBERT [5], RoBERTa [6]) · **decoder** (GPT-2 [7]) · **encoder–decoder** (BART [8], T5 [9]). Không phải lớp nào cũng có cho mọi họ — ví dụ DistilBERT chỉ có encoder nên không có `…ForSeq2SeqLM`. Danh sách đầy đủ: tài liệu Auto Classes [40].

In [ ]:
from transformers import DistilBertModel, DistilBertForSequenceClassification

base_model = DistilBertModel.from_pretrained("distilbert-base-cased")                 # chỉ thân
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-cased",   # thân + head mới
                                                           num_labels=2)
# Cảnh báo "Some weights ... newly initialized: ['classifier...', 'pre_classifier...']" là BÌNH THƯỜNG:
# đó chính là head chưa được huấn luyện.
n_body = sum(p.numel() for p in base_model.parameters())
n_all = sum(p.numel() for p in model.parameters())
print(f"Tham số thân: {n_body:,} | thân+head: {n_all:,} | head: {n_all-n_body:,}")

In [ ]:
# Khởi tạo từ config với TRỌNG SỐ NGẪU NHIÊN (dùng khi pre-train từ đầu hoặc thử kiến trúc)
from transformers import DistilBertConfig
config = DistilBertConfig(num_labels=2, n_layers=2)          # thu nhỏ còn 2 layer để thử nhanh
random_model = DistilBertForSequenceClassification(config)
print(f"{random_model.config.n_layers} layers | {sum(p.numel() for p in random_model.parameters()):,} tham số")

### Truyền dữ liệu vào model và tính loss
Model nhận **tham số từ khoá**; các key tokenizer trả về (`input_ids`, `attention_mask`, …) trùng tên tham số của `forward`, nên chỉ cần `model(**inputs)`.
Nếu truyền thêm `labels`, model tự tính loss phù hợp:
- `num_labels == 1` → hồi quy (MSE);
- nhãn là số nguyên → phân loại đơn nhãn (cross-entropy);
- `problem_type="multi_label_classification"` → đa nhãn (BCE với sigmoid từng lớp).

*Vì sao phân loại nhị phân lại có 2 logit?* Để dùng chung cross-entropy/softmax cho mọi số lớp. Thêm một cột trọng số không làm giảm chất lượng; tương đương về toán học với 1 logit + sigmoid.

In [ ]:
model_inputs = tokenizer(input_str, return_tensors="pt")
out = model(**model_inputs)
print("Logits:", out.logits, "| softmax:", out.logits.softmax(-1))

# Cách 1: tự tính loss như PyTorch thông thường
label = torch.tensor([1])
loss = F.cross_entropy(out.logits, label)
loss.backward()
print("Loss tự tính:", loss.item())

# Cách 2: để model tự tính khi có 'labels'
out2 = model(**model_inputs, labels=label)
print("Loss model tính:", out2.loss.item())
model.zero_grad()

### Hidden states và attention — nguyên liệu cho project phân tích
`output_hidden_states=True` trả về biểu diễn ở **mọi tầng** (tầng 0 = embedding), `output_attentions=True` trả về trọng số attention kích thước `(batch, heads, query, key)` cho từng tầng. Đây là công cụ chính của các nghiên cứu probing như *What does BERT look at?* [13].
Lưu ý: muốn lấy attention, dùng `attn_implementation="eager"` vì các kernel tối ưu (SDPA/FlashAttention [23]) không trả ma trận attention.

In [ ]:
from transformers import AutoModel

viz_model = AutoModel.from_pretrained("distilbert-base-cased", output_attentions=True,
                                      output_hidden_states=True, attn_implementation="eager").eval()
model_inputs = tokenizer(input_str, return_tensors="pt")
with torch.no_grad():
    viz_out = viz_model(**model_inputs)
tokens = tokenizer.convert_ids_to_tokens(model_inputs.input_ids[0])
print("Số tầng hidden (gồm embedding):", len(viz_out.hidden_states), "| mỗi tầng:", tuple(viz_out.hidden_states[0].shape))
print("Số tầng attention:", len(viz_out.attentions), "| mỗi tầng:", tuple(viz_out.attentions[0].shape))

n_layers, n_heads, T = len(viz_out.attentions), viz_out.attentions[0].shape[1], len(tokens)
fig, axes = plt.subplots(n_layers, n_heads, figsize=(2.2 * n_heads, 2.2 * n_layers))
for l in range(n_layers):
    for h in range(n_heads):
        ax = axes[l, h]
        ax.imshow(viz_out.attentions[l][0, h], cmap="viridis")
        ax.set_xticks(range(T)); ax.set_yticks(range(T))
        ax.set_xticklabels(tokens if l == n_layers - 1 else [], rotation=90, fontsize=6)
        ax.set_yticklabels(tokens if h == 0 else [], fontsize=6)
        if l == n_layers - 1: ax.set_xlabel(f"head {h}", fontsize=7)
        if h == 0: ax.set_ylabel(f"layer {l}", fontsize=7)
plt.suptitle("Attention của DistilBERT (trục y = query, trục x = key)")
plt.tight_layout(); plt.show()

### Suy luận hiệu quả (thói quen nên có)
| Kỹ thuật | Cách làm | Tác dụng |
|---|---|---|
| Tắt gradient | `with torch.inference_mode():` | Giảm bộ nhớ, nhanh hơn |
| Nửa độ chính xác | `from_pretrained(..., dtype=torch.float16)` (GPU) | Giảm ½ bộ nhớ [24] |
| Attention tối ưu | `attn_implementation="sdpa"` (mặc định với nhiều mô hình) | Dùng kernel kiểu FlashAttention [23] |
| Tự phân bổ thiết bị | `device_map="auto"` (cần `accelerate`) | Chạy mô hình lớn trên nhiều GPU/CPU |
| Lượng tử hoá 4/8-bit | `BitsAndBytesConfig` | Chạy LLM 7B trên GPU 16GB |

# PHẦN 1 — FINE-TUNING

> **⏱ Ý chính.** Fine-tune = tải checkpoint + head mới → token hoá dữ liệu bằng `datasets.map` → huấn luyện bằng vòng lặp PyTorch *hoặc* `Trainer` → đánh giá → lưu. Mô hình HF chính là `torch.nn.Module`, nên mọi kỹ thuật PyTorch đều dùng được.

## 1.1 Tải dữ liệu với `datasets`
Hub chứa hàng trăm nghìn bộ dữ liệu. Ví dụ dùng **IMDB** — 50.000 bình luận phim, nhãn tích cực/tiêu cực [11]. Thư viện `datasets` [2] lưu dữ liệu dạng Apache Arrow (ánh xạ bộ nhớ) nên xử lý được bộ dữ liệu lớn hơn RAM; `.map(batched=True)` xử lý theo lô và **tự cache** kết quả.

In [ ]:
from datasets import load_dataset, DatasetDict

imdb = load_dataset("stanfordnlp/imdb")
print(imdb)
print(imdb["train"].features)

def truncate(example):
    """Giữ 50 từ đầu để chạy nhanh trên CPU/Colab."""
    return {"text": " ".join(example["text"].split()[:50]), "label": example["label"]}

# 128 mẫu train, 32 mẫu validation (lấy từ tập train đã xáo trộn — tập test để dành cho đánh giá cuối)
shuffled = imdb["train"].shuffle(seed=1111)
small_imdb = DatasetDict(
    train=shuffled.select(range(128)).map(truncate),
    val=shuffled.select(range(128, 160)).map(truncate),
)
print(small_imdb)
print("Phân bố nhãn train:", Counter(small_imdb["train"]["label"]))
small_imdb["train"][:3]

In [ ]:
# Token hoá KHÔNG padding ở đây — padding động sẽ làm trong DataCollator theo từng batch
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")
tokenized = small_imdb.map(lambda b: tokenizer(b["text"], truncation=True, max_length=128),
                           batched=True, remove_columns=["text"])
tokenized = tokenized.rename_column("label", "labels")
print(tokenized)
print("Độ dài 5 mẫu đầu:", [len(x) for x in tokenized["train"][:5]["input_ids"]])

## 1.2 Huấn luyện bằng vòng lặp PyTorch

- **AdamW** [17]: Adam với *weight decay tách rời* khỏi cập nhật gradient — lựa chọn mặc định cho Transformer.
- **Linear schedule with warmup**: tăng LR tuyến tính trong vài trăm bước đầu rồi giảm tuyến tính về 0; warmup giúp tránh các cập nhật quá lớn phá hỏng trọng số tiền huấn luyện [4, 18].
- Các lựa chọn khác: scheduler của Hugging Face [40], của PyTorch (ví dụ `ReduceLROnPlateau`), hoặc tự viết.

> ⚠️ **Lỗi trong bản gốc đã sửa:** biến `loss` dùng ở vòng validation chưa được khởi tạo (gây `NameError`); dữ liệu chưa được chuyển sang GPU; chưa đo accuracy.

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm.auto import tqdm

collator = DataCollatorWithPadding(tokenizer)       # padding động theo batch
train_dl = DataLoader(tokenized["train"], batch_size=16, shuffle=True, collate_fn=collator)
val_dl = DataLoader(tokenized["val"], batch_size=32, collate_fn=collator)

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-cased", num_labels=2).to(device)
num_epochs = 3
num_steps = num_epochs * len(train_dl)
optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * num_steps),
                                            num_training_steps=num_steps)

best_val_loss = float("inf")
bar = tqdm(range(num_steps))
for epoch in range(num_epochs):
    model.train()
    for batch in train_dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        optimizer.zero_grad()
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # gradient clipping
        optimizer.step(); scheduler.step(); bar.update(1)

    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch in val_dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            val_loss += out.loss.item()
            correct += (out.logits.argmax(-1) == batch["labels"]).sum().item()
            total += batch["labels"].numel()
    avg = val_loss / len(val_dl)
    print(f"epoch {epoch}: val_loss={avg:.4f} | val_acc={correct/total:.3f}")
    if avg < best_val_loss:
        best_val_loss = avg
        model.save_pretrained("checkpoints/pytorch_loop_best")   # lưu theo chuẩn HF
        tokenizer.save_pretrained("checkpoints/pytorch_loop_best")
        print("  → đã lưu checkpoint tốt nhất")

## 1.3 Huấn luyện bằng `Trainer`

`TrainingArguments` gom mọi siêu tham số (LR, batch, số epoch, lịch đánh giá/lưu, logging, mixed precision…) — xem danh sách đầy đủ trong tài liệu [40]. `Trainer` thực hiện vòng lặp, đánh giá, lưu checkpoint, phân tán nhiều GPU. Một số tuỳ chọn hay dùng:

| Tham số | Ý nghĩa |
|---|---|
| `eval_strategy`, `save_strategy` | Khi nào đánh giá/lưu (`"epoch"`, `"steps"`) |
| `load_best_model_at_end` + `metric_for_best_model` | Tự nạp checkpoint tốt nhất sau khi train |
| `fp16` / `bf16` | Huấn luyện nửa độ chính xác [24] |
| `gradient_accumulation_steps` | Mô phỏng batch lớn khi GPU nhỏ |
| `save_total_limit` | Giới hạn số checkpoint trên đĩa |
| `report_to` | `"none"`, `"tensorboard"`, `"wandb"`, `"mlflow"` |

`compute_metrics` nhận `(logits, labels)` ở cuối mỗi lần đánh giá. Thư viện `evaluate` cung cấp sẵn accuracy, F1, seqeval, ROUGE…

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate

acc_metric, f1_metric = evaluate.load("accuracy"), evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
            "macro_f1": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]}

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-cased", num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"}, label2id={"NEGATIVE": 0, "POSITIVE": 1})

args = TrainingArguments(
    output_dir="sample_hf_trainer",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)
trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["val"],          # đổi sang tập test khi đánh giá cuối cùng!
    processing_class=tokenizer,             # thay cho tham số cũ tokenizer=
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

### Callback: logging và early stopping
Callback cho phép chèn hành động vào các mốc huấn luyện (`on_log`, `on_evaluate`, `on_epoch_end`…) [40]. `EarlyStoppingCallback` dừng khi metric không cải thiện sau `patience` lần đánh giá — cần `load_best_model_at_end=True` và `metric_for_best_model`.

In [ ]:
from transformers import TrainerCallback, EarlyStoppingCallback

class JsonlLoggingCallback(TrainerCallback):
    """Ghi mọi log (loss, metric, lr) ra file .jsonl để vẽ biểu đồ sau."""
    def __init__(self, log_path):
        self.log_path = log_path
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.is_local_process_zero and logs:
            logs = {k: v for k, v in logs.items() if k != "total_flos"}
            with open(self.log_path, "a") as f:
                f.write(json.dumps({"step": state.global_step, **logs}) + "\n")

trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=2))
trainer.add_callback(JsonlLoggingCallback("sample_hf_trainer/log.jsonl"))
trainer.train()

In [ ]:
# Đánh giá và dự đoán
print(trainer.evaluate())                        # chỉ metric
results = trainer.predict(tokenized["val"])      # metric + logits + nhãn
print(results.metrics)
print("Logits 3 mẫu đầu:\n", results.predictions[:3])

# Lưu mô hình tốt nhất vào đường dẫn CỐ ĐỊNH (bản gốc dùng 'checkpoint-8' cứng → dễ sai khi đổi batch/epoch)
print("Checkpoint tốt nhất:", trainer.state.best_model_checkpoint)
trainer.save_model("sample_hf_trainer/best")     # lưu cả model lẫn tokenizer (processing_class)

finetuned = AutoModelForSequenceClassification.from_pretrained("sample_hf_trainer/best").eval()
for s in ["I enjoyed the movie!", "What a waste of two hours."]:
    with torch.no_grad():
        logits = finetuned(**tokenizer(s, return_tensors="pt")).logits
    print(f"{s!r:35s} → {finetuned.config.id2label[int(logits.argmax())]}")

## 1.4 Mẹo fine-tune thực tế

**Siêu tham số khởi đầu tốt** (theo Devlin et al. [4] và kinh nghiệm cộng đồng [38]):
- Số epoch: {2, 3, 4} — dữ liệu càng nhiều càng cần ít epoch.
- Batch size: lớn nhất GPU cho phép (dùng `gradient_accumulation_steps` nếu thiếu bộ nhớ).
- Optimizer AdamW [17], LR ∈ {2e-5, 3e-5, 5e-5}; warmup 6–10% số bước; weight decay ∈ {0, 0.01, 0.1}.

**Ổn định kết quả.** Fine-tune BERT trên dữ liệu nhỏ dao động mạnh theo *seed* (khởi tạo head và thứ tự dữ liệu) [19]. Mosbach et al. [18] chỉ ra nguyên nhân chính là *tối ưu hoá* (gradient biến mất đầu quá trình), khuyến nghị: dùng bias correction của Adam, LR nhỏ, **train nhiều bước hơn**. → Luôn báo cáo **trung bình ± độ lệch chuẩn trên ≥ 3 seed**.

**Theo dõi validation loss/metric** để chọn checkpoint và siêu tham số; không bao giờ chọn trên tập test.

## 1.5 Fine-tune tiết kiệm tham số với LoRA (PEFT)

> **⏱ Ý chính.** LoRA đóng băng trọng số gốc $W_0$ và chỉ học cập nhật hạng thấp $\Delta W = BA$ với $r \ll d$ [21]. Thư viện `peft` [22] bọc mô hình HF chỉ với vài dòng.

$$W' = W_0 + \frac{\alpha}{r} BA,\qquad A\in\mathbb R^{r\times d_{in}},\ B\in\mathbb R^{d_{out}\times r}$$

Lưu ý với DistilBERT: tên module attention là `q_lin`, `v_lin`; head gồm `pre_classifier` và `classifier` đều **khởi tạo mới** nên phải để chúng **được huấn luyện** qua `modules_to_save` (nếu không, `pre_classifier` ngẫu nhiên sẽ bị đóng băng).

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
                      target_modules=["q_lin", "v_lin"],
                      modules_to_save=["pre_classifier", "classifier"])
lora_model = get_peft_model(
    AutoModelForSequenceClassification.from_pretrained("distilbert-base-cased", num_labels=2), lora_cfg)
lora_model.print_trainable_parameters()

lora_args = TrainingArguments(output_dir="lora_trainer", per_device_train_batch_size=16,
                              num_train_epochs=4, learning_rate=5e-4,     # LoRA thường cần LR lớn hơn
                              eval_strategy="epoch", save_strategy="no", logging_strategy="epoch",
                              fp16=torch.cuda.is_available(), report_to="none", seed=SEED)
lora_trainer = Trainer(model=lora_model, args=lora_args, train_dataset=tokenized["train"],
                       eval_dataset=tokenized["val"], processing_class=tokenizer,
                       data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
lora_trainer.train()
lora_model.save_pretrained("lora_adapter")      # chỉ lưu adapter: vài MB thay vì ~260MB
print("Kích thước adapter:", sum(os.path.getsize(os.path.join("lora_adapter", f)) for f in os.listdir("lora_adapter")) // 1024, "KB")

# PHẦN 2 — SINH VĂN BẢN, DỮ LIỆU TUỲ CHỈNH, PIPELINE, MLM

## 2.1 Sinh văn bản với `generate`

> **⏱ Ý chính.** Mô hình causal LM sinh từng token; `generate()` điều khiển *cách chọn token*. Dùng `max_new_tokens` (không phải `max_length`, vốn tính cả prompt). Khi sinh theo batch với decoder-only, **padding phải ở bên trái**.

| Tham số | Tác dụng |
|---|---|
| `do_sample=False` | Greedy (hoặc beam search với `num_beams>1`) |
| `temperature` | <1 an toàn, >1 đa dạng |
| `top_k`, `top_p` | Lấy mẫu trong *k* token tốt nhất / nucleus có tổng xác suất ≥ *p* [20] |
| `repetition_penalty`, `no_repeat_ngram_size` | Chống lặp |
| `num_return_sequences` | Sinh nhiều phương án cùng lúc |

In [ ]:
from transformers import AutoModelForCausalLM, GenerationConfig

gpt2_tok = AutoTokenizer.from_pretrained("distilgpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device).eval()
gpt2_tok.pad_token = gpt2_tok.eos_token          # GPT-2 không có pad token

prompt = "Once upon a time"
inp = gpt2_tok(prompt, return_tensors="pt").to(device)
gen_cfg = GenerationConfig(max_new_tokens=40, do_sample=True, top_p=0.9, temperature=0.8,
                           num_return_sequences=5, pad_token_id=gpt2_tok.eos_token_id)
torch.manual_seed(SEED)
with torch.inference_mode():
    outs = gpt2.generate(**inp, generation_config=gen_cfg)
for i, o in enumerate(gpt2_tok.batch_decode(outs, skip_special_tokens=True), 1):
    print(f"{i}) {o}")

In [ ]:
# Sinh theo batch với decoder-only: BẮT BUỘC padding bên trái
prompts = ["The weather today is", "Deep learning is a field that"]
for side in ["right", "left"]:
    gpt2_tok.padding_side = side
    b = gpt2_tok(prompts, return_tensors="pt", padding=True).to(device)
    with torch.inference_mode():
        o = gpt2.generate(**b, max_new_tokens=12, do_sample=False, pad_token_id=gpt2_tok.eos_token_id)
    print(f"\npadding_side={side}:")
    for t in gpt2_tok.batch_decode(o, skip_special_tokens=True):
        print("  ", t)
print("\nVới padding phải, câu ngắn phải sinh tiếp SAU các token đệm → kết quả lệch. Luôn dùng padding trái.")

## 2.2 Chat template — cách "nói chuyện" đúng với LLM đã instruction-tuned

> **⏱ Ý chính.** Mỗi LLM hội thoại được huấn luyện với một **định dạng hội thoại riêng** (token vai trò, token kết thúc lượt). Tự ghép chuỗi bằng tay dễ sai; dùng `tokenizer.apply_chat_template()` để định dạng đúng như lúc huấn luyện [40].

Ví dụ dùng **Qwen2.5-0.5B-Instruct** [37] — nhỏ, chạy được cả CPU, hỗ trợ đa ngữ.

In [ ]:
chat_name = "Qwen/Qwen2.5-0.5B-Instruct"
chat_tok = AutoTokenizer.from_pretrained(chat_name)
chat_model = AutoModelForCausalLM.from_pretrained(
    chat_name, dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()

messages = [
    {"role": "system", "content": "You are a concise NLP teaching assistant."},
    {"role": "user", "content": "In two sentences, what is the difference between BERT and GPT?"},
]
prompt_text = chat_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("Chuỗi thật sự đưa vào mô hình:\n", prompt_text)

ids = chat_tok(prompt_text, return_tensors="pt").to(device)
with torch.inference_mode():
    out = chat_model.generate(**ids, max_new_tokens=120, do_sample=False)
print("Trả lời:\n", chat_tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True))

## 2.3 Dataset tuỳ chỉnh (ví dụ bài toán sinh có điều kiện)

Dùng **E2E NLG** [12]: ánh xạ *biểu diễn ý nghĩa có cấu trúc* (meaning representation, MR) về nhà hàng thành *mô tả ngôn ngữ tự nhiên*. Dữ liệu lấy từ bản Parquet tự chuyển đổi trên Hub (`tuetschek/e2e_nlg`) [40].

> ⚠️ **Sửa lỗi bản gốc:** bản gốc đặt `human_reference` làm đầu vào và `meaning_representation` làm nhãn — **ngược** với định nghĩa bài toán. Bản dưới đây: đầu vào = MR, nhãn = câu mô tả.

In [ ]:
from datasets import Dataset
E2E_PATH = "hf://datasets/tuetschek/e2e_nlg@refs%2Fconvert%2Fparquet/default/train/0000.parquet"

# Cách 1: nạp vào HF Datasets
df = pd.read_parquet(E2E_PATH)
e2e = Dataset.from_pandas(df)
print(e2e)
print("MR   :", e2e[0]["meaning_representation"])
print("Text :", e2e[0]["human_reference"])

In [ ]:
# Cách 2: torch Dataset tự viết (token hoá "lười" khi __getitem__ được gọi)
from torch.utils.data import Dataset as TorchDataset
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

class E2EDataset(TorchDataset):
    def __init__(self, path, tokenizer, max_len=128):
        self.data = pd.read_parquet(path)[["meaning_representation", "human_reference"]].to_dict("records")
        self.tok, self.max_len = tokenizer, max_len
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        ex = self.data[i]
        # text_target → tokenizer tự tạo trường 'labels' cho seq2seq
        return self.tok(ex["meaning_representation"], text_target=ex["human_reference"],
                        truncation=True, max_length=self.max_len)

bart_tok = AutoTokenizer.from_pretrained("facebook/bart-base")
e2e_ds = E2EDataset(E2E_PATH, bart_tok)
print(e2e_ds[0])

# Collator cho seq2seq: pad cả input lẫn labels (labels đệm bằng -100 để bỏ qua khi tính loss)
bart = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-base")
s2s_collator = DataCollatorForSeq2Seq(bart_tok, model=bart)
b = s2s_collator([e2e_ds[i] for i in range(4)])
print({k: tuple(v.shape) for k, v in b.items()})
print("Loss trước fine-tune:", bart(**b).loss.item())

## 2.4 Pipeline — dùng mô hình có sẵn trong một dòng

`pipeline(task, model=...)` gói trọn tokenizer + model + hậu xử lý [1]. Rất tiện để thử nhanh, làm baseline, hoặc demo; trong nghiên cứu thường vẫn cần kiểm soát từng bước.

| Task | Ví dụ checkpoint |
|---|---|
| `sentiment-analysis` / `text-classification` | `siebert/sentiment-roberta-large-english` [10] |
| `fill-mask` | `bert-base-cased` [4] |
| `token-classification` (NER) | `dslim/bert-base-NER` |
| `question-answering` | `deepset/roberta-base-squad2` |
| `summarization` | `facebook/bart-large-cnn` [8] |
| `text-generation` | `Qwen/Qwen2.5-0.5B-Instruct` [37] |
| `feature-extraction` | bất kỳ encoder nào |

In [ ]:
from transformers import pipeline
dev_id = 0 if device == "cuda" else -1

sentiment = pipeline("sentiment-analysis", model="siebert/sentiment-roberta-large-english", device=dev_id)
print(sentiment("Hugging Face Transformers is really cool!"))
print(sentiment(["I didn't know if I would like Hákarl, but it turned out pretty good.",
                 "I didn't know if I would like Hákarl, and it was just as bad as I'd heard."]))

ner = pipeline("token-classification", model="dslim/bert-base-NER", aggregation_strategy="simple", device=dev_id)
print(ner("Hoang Van Quy teaches at Thuyloi University in Hanoi."))

## 2.5 Masked Language Modeling

BERT được tiền huấn luyện bằng cách che ~15% token và đoán lại [4]. Có hai cách dùng: qua `pipeline("fill-mask")` hoặc tự lấy logits tại vị trí `[MASK]`.

In [ ]:
from transformers import AutoModelForMaskedLM

mlm_tok = AutoTokenizer.from_pretrained("bert-base-cased")          # (bản gốc dùng fast=True — không phải tham số hợp lệ)
bert = AutoModelForMaskedLM.from_pretrained("bert-base-cased").eval()

prompt = "I am [MASK] to learn about HuggingFace!"
print(pipeline("fill-mask", model=bert, tokenizer=mlm_tok)(prompt))

inputs = mlm_tok(prompt, return_tensors="pt")
mask_pos = (inputs.input_ids[0] == mlm_tok.mask_token_id).nonzero(as_tuple=True)[0]
with torch.inference_mode():
    logits = bert(**inputs).logits[0, mask_pos]
top5 = logits.softmax(-1).topk(5)
print("\n" + prompt)
for p, i in zip(top5.values[0], top5.indices[0]):
    print(f"  {mlm_tok.decode(int(i)):12s} {p:.3f}")

# PHẦN 3 — ỨNG DỤNG HUGGING FACE CHO TIẾNG VIỆT 🇻🇳

> **⏱ Ý chính.** Ba điều khác biệt khi làm tiếng Việt:
> 1. **Unicode và dấu thanh**: cùng một chữ có nhiều cách mã hoá (NFC/NFD) và hai kiểu đặt dấu (hoà/hòa) → phải chuẩn hoá trước khi token hoá.
> 2. **Âm tiết ≠ từ**: khoảng trắng tách âm tiết ("Hà Nội" là 1 từ, 2 âm tiết). **PhoBERT bắt buộc đầu vào đã tách từ** (`Hà_Nội`) [28, 29].
> 3. **Tokenizer quyết định chi phí**: tokenizer tiếng Anh chẻ tiếng Việt thành rất nhiều token → tốn ngữ cảnh, tốn tiền, kém chất lượng [26, 27].

## 3.1 Chọn mô hình tiếng Việt trên Hub

| Nhóm | Checkpoint | Ghi chú | Tài liệu |
|---|---|---|---|
| Encoder văn bản chuẩn | `vinai/phobert-base`, `vinai/phobert-base-v2`, `vinai/phobert-large` | RoBERTa, **cần tách từ**, tối đa **256** token, tokenizer **slow** | [28] |
| Encoder mạng xã hội | `uitnlp/visobert` | Học trên dữ liệu MXH, **không cần tách từ**, tốt với teencode/emoji | [30] |
| Encoder đa ngữ | `FacebookAI/xlm-roberta-base/large` | 100 ngôn ngữ, fast tokenizer, không cần tách từ | [25] |
| Seq2seq | `VietAI/vit5-base`, `vinai/bartpho-syllable`, `vinai/bartpho-word` | Tóm tắt, sinh tiêu đề, khôi phục dấu | [31, 32] |
| Embedding (tìm kiếm) | `intfloat/multilingual-e5-small/base`, `BAAI/bge-m3` | e5 cần tiền tố `query:` / `passage:` | [35, 36] |
| LLM hội thoại | `Qwen/Qwen2.5-*-Instruct` và các LLM đa ngữ khác | Kiểm tra chất lượng tiếng Việt trước khi chọn | [37] |

**Dữ liệu tiếng Việt tiêu biểu:** UIT-VSFC (phản hồi sinh viên: cảm xúc + chủ đề) [33], PhoNER_COVID19 (NER, có bản âm tiết và bản đã tách từ) [34].

In [ ]:
# ============================================================
# 3.2 CHUẨN HOÁ TIẾNG VIỆT: NFC + KIỂU ĐẶT DẤU + TEENCODE
# ============================================================
TONE_MARKS = {"\u0301", "\u0300", "\u0309", "\u0303", "\u0323"}   # sắc, huyền, hỏi, ngã, nặng
VALID_PAIRS = {("o", "a"), ("o", "e"), ("u", "y")}
_pat = re.compile(r"(?<![qQ])([oOuU])([^\W\d_])(?![^\W\d_])")

def normalize_tone_style(text: str) -> str:
    """Thống nhất về kiểu đặt dấu cũ: hoá→hóa, thuỷ→thủy, xoè→xòe.
    Không đụng âm tiết có phụ âm cuối (hoán, khoảng) và cụm 'qu' (quý)."""
    def repl(m):
        first, second = m.group(1), m.group(2)
        nfd = unicodedata.normalize("NFD", second)
        base, marks = nfd[0], nfd[1:]
        if (first.lower(), base.lower()) not in VALID_PAIRS or len(marks) != 1 or marks not in TONE_MARKS:
            return m.group(0)
        return unicodedata.normalize("NFC", first + marks) + base
    return _pat.sub(repl, text)

TEENCODE = {"ko": "không", "k0": "không", "khum": "không", "hok": "không", "dc": "được", "đc": "được",
            "vs": "với", "mn": "mọi người", "ntn": "như thế nào", "j": "gì", "bik": "biết", "thik": "thích"}

def normalize_vi(text: str, fix_teencode: bool = True) -> str:
    text = unicodedata.normalize("NFC", text)            # 1) Unicode dựng sẵn
    text = normalize_tone_style(text)                     # 2) một kiểu đặt dấu
    text = re.sub(r"(\w)\1{2,}", r"\1", text)             # 3) "hayyyy" → "hay"
    if fix_teencode:                                      # 4) teencode (từ điển nhỏ, cần mở rộng)
        text = re.sub(r"\b\w+\b", lambda m: TEENCODE.get(m.group(0).lower(), m.group(0)), text)
    return re.sub(r"\s+", " ", text).strip()

def remove_accents(text: str) -> str:
    text = "".join(c for c in unicodedata.normalize("NFD", text) if unicodedata.category(c) != "Mn")
    return text.replace("đ", "d").replace("Đ", "D")

a, b = "Thuỷ lợi", unicodedata.normalize("NFD", "Thủy lợi")
print(f"'{a}' vs '{b}': bằng nhau? {a == b} | code point: {len(a)} vs {len(b)}")
xlmr_tok = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
print("XLM-R token hoá khác nhau:", xlmr_tok.tokenize(a), "vs", xlmr_tok.tokenize(b))
print("Sau chuẩn hoá bằng nhau?", normalize_vi(a) == normalize_vi(b), "→", normalize_vi(a))
for t in ["Hoà bình", "quý khách", "khoảng cách", "phim hayyyy mà ko rẻ, j z trời"]:
    print(f"  {t:32s} → {normalize_vi(t)}")

In [ ]:
# ============================================================
# 3.3 TÁCH TỪ (WORD SEGMENTATION) CHO PhoBERT
# ============================================================
# PhoBERT được tiền huấn luyện trên văn bản tách từ bằng VnCoreNLP/RDRSegmenter [28, 29].
# Trên Colab, underthesea/pyvi dễ cài (không cần Java) và cho kết quả gần tương đương.
# Với kết quả để công bố, nên dùng đúng VnCoreNLP để khớp phân phối tiền huấn luyện.
def word_segment(text: str) -> str:
    try:
        from underthesea import word_tokenize
        return word_tokenize(text, format="text")
    except Exception:
        from pyvi import ViTokenizer
        return ViTokenizer.tokenize(text)

for s in ["Trường Đại học Thủy lợi đào tạo kỹ sư tài nguyên nước.",
          "TS. Hoàng Văn Quý giảng dạy môn học sâu tại Hà Nội."]:
    print(s, "\n  →", word_segment(normalize_vi(s)))

## 3.4 So sánh tokenizer trên tiếng Việt — fertility và "thuế token"

**Fertility** = số token trung bình trên một đơn vị ngôn ngữ (ở đây: âm tiết) [26]. Fertility cao nghĩa là: ngữ cảnh 512 token chứa ít văn bản hơn, huấn luyện/suy luận chậm hơn, và khi dùng API LLM tính phí theo token thì **người dùng tiếng Việt trả nhiều tiền hơn cho cùng nội dung** — hiện tượng Petrov et al. gọi là *bất công bằng do tokenizer* [27].

In [ ]:
pairs_vi_en = [
    ("Biến đổi khí hậu làm gia tăng xâm nhập mặn ở Đồng bằng sông Cửu Long.",
     "Climate change increases saltwater intrusion in the Mekong Delta."),
    ("Sinh viên cần nộp báo cáo thực hành trước thứ Sáu tuần sau.",
     "Students must submit the lab report before next Friday."),
    ("Hệ thống cảnh báo sạt lở bờ sông sử dụng cảm biến IoT và học sâu.",
     "The riverbank landslide warning system uses IoT sensors and deep learning."),
]
vi = [normalize_vi(v) for v, _ in pairs_vi_en]; en = [e for _, e in pairs_vi_en]
n_syll = sum(len(s.split()) for s in vi)

candidates = {
    "GPT-2 (byte BPE, tiếng Anh)":  ("openai-community/gpt2", False),
    "BERT-base-cased (WordPiece)":  ("bert-base-cased", False),
    "mBERT (WordPiece đa ngữ)":     ("bert-base-multilingual-cased", False),
    "XLM-R (SentencePiece)":        ("FacebookAI/xlm-roberta-base", False),
    "PhoBERT (BPE, đã tách từ)":    ("vinai/phobert-base", True),
    "Qwen2.5 (byte BPE đa ngữ)":    ("Qwen/Qwen2.5-0.5B-Instruct", False),
}
rows = []
for label, (ckpt, need_seg) in candidates.items():
    try:
        t = AutoTokenizer.from_pretrained(ckpt)
        vi_in = [word_segment(s) for s in vi] if need_seg else vi
        n_vi = sum(len(t.tokenize(s)) for s in vi_in)
        n_en = sum(len(t.tokenize(s)) for s in en)
        rows.append({"Tokenizer": label, "fast": t.is_fast, "Token VI": n_vi,
                     "Token/âm tiết": round(n_vi / n_syll, 2), "Token EN": n_en,
                     "VI/EN": round(n_vi / n_en, 2), "Ví dụ": " ".join(t.tokenize(vi_in[0])[:10])})
    except Exception as e:
        rows.append({"Tokenizer": label, "Ví dụ": f"lỗi tải: {str(e)[:50]}"})
pd.set_option("display.max_colwidth", 90)
display(pd.DataFrame(rows).sort_values("Token/âm tiết"))
print(f"Tổng số âm tiết tiếng Việt: {n_syll}")
print("Cột VI/EN > 1 nghĩa là cùng một nội dung, tiếng Việt tốn nhiều token hơn tiếng Anh.")

**Đọc kết quả (điển hình):** GPT-2 và BERT tiếng Anh chẻ tiếng Việt thành 3–4 token/âm tiết và tỉ lệ VI/EN > 2; PhoBERT < 1 token/âm tiết vì mỗi token là một *từ* đã tách; XLM-R và Qwen2.5 ở mức trung gian. Cột `fast` cho thấy PhoBERT chỉ có tokenizer slow → cần căn nhãn NER thủ công (mục 3.7).

In [ ]:
# ============================================================
# 3.5 PIPELINE TIẾNG VIỆT: FILL-MASK, CẢM XÚC, TÓM TẮT
# ============================================================
# (a) PhoBERT fill-mask: so sánh đầu vào có/không tách từ
pho_mlm = pipeline("fill-mask", model="vinai/phobert-base", top_k=5, device=dev_id)
for raw, seg in [("Hà Nội là <mask> của Việt Nam .", "Hà_Nội là <mask> của Việt_Nam ."),
                 ("Mùa lũ năm nay gây <mask> lở nghiêm trọng .", "Mùa lũ năm nay gây <mask> lở nghiêm_trọng .")]:
    print("Không tách từ:", [(r["token_str"], round(r["score"], 2)) for r in pho_mlm(raw)])
    print("Có tách từ   :", [(r["token_str"], round(r["score"], 2)) for r in pho_mlm(seg)], "\n")

# (b) Phân tích cảm xúc bằng checkpoint cộng đồng đã fine-tune từ PhoBERT (đầu vào cần tách từ)
try:
    vi_sent = pipeline("text-classification", model="wonrax/phobert-base-vietnamese-sentiment", device=dev_id)
    for s in ["Thầy giảng bài rất dễ hiểu và nhiệt tình.", "Phòng học nóng, máy chiếu hỏng liên tục.",
              "Không phải là môn học dở."]:
        print(f"{s:45s} → {vi_sent(word_segment(normalize_vi(s)))[0]}")
except Exception as e:
    print("Không tải được mô hình cảm xúc:", str(e)[:80])

In [ ]:
# (c) Tóm tắt tiếng Việt với ViT5 đã fine-tune trên VietNews [31]
article = ("Trường Đại học Thủy lợi vừa triển khai hệ thống giám sát sạt lở bờ sông sử dụng cảm biến IoT "
           "tại một số tỉnh Đồng bằng sông Cửu Long. Hệ thống đo độ nghiêng, độ ẩm đất và mực nước theo thời gian thực, "
           "kết hợp mô hình học sâu để dự báo nguy cơ sạt lở trước từ 24 đến 48 giờ. Theo nhóm nghiên cứu, "
           "dữ liệu được truyền về trung tâm điều hành qua mạng LoRaWAN và hiển thị trên bản đồ số, giúp chính quyền "
           "địa phương chủ động sơ tán người dân khi có cảnh báo.")
for ckpt in ["VietAI/vit5-base-vietnews-summarization", "VietAI/vit5-large-vietnews-summarization"]:
    try:
        s2s_tok = AutoTokenizer.from_pretrained(ckpt)
        s2s = AutoModelForSeq2SeqLM.from_pretrained(ckpt).to(device).eval()
        # Theo model card: thêm tiền tố "vietnews: " và hậu tố " </s>"
        enc = s2s_tok("vietnews: " + normalize_vi(article) + " </s>", return_tensors="pt",
                      truncation=True, max_length=512).to(device)
        with torch.inference_mode():
            out = s2s.generate(**enc, max_new_tokens=80, num_beams=4, early_stopping=True)
        print(f"[{ckpt}]\n", s2s_tok.decode(out[0], skip_special_tokens=True))
        break
    except Exception as e:
        print(f"Không dùng được {ckpt}: {str(e)[:80]}")

## 3.6 Fine-tune PhoBERT phân loại cảm xúc với `Trainer`

Quy trình chuẩn cho tiếng Việt: **chuẩn hoá → tách từ → token hoá (tối đa 256) → `Trainer` → phân tích lỗi**.
Cell dưới thử tải **UIT-VSFC** [33]; nếu không tải được (ví dụ repo dùng loading script không còn được `datasets` hỗ trợ) thì dùng bộ dữ liệu mini dựng sẵn. Đổi `MODEL` sang `uitnlp/visobert` hoặc `FacebookAI/xlm-roberta-base` để so sánh (hai mô hình này không cần tách từ).

In [ ]:
from datasets import Dataset as HFDataset

def load_vi_sentiment(n_train=3000):
    try:
        ds = load_dataset("uitnlp/vietnamese_students_feedback")
        ds = ds.filter(lambda x: x["sentiment"] != 1)                  # bỏ trung tính → bài toán nhị phân
        ds = ds.map(lambda x: {"text": x["sentence"], "label": 1 if x["sentiment"] == 2 else 0})
        print("Đã tải UIT-VSFC.")
        return DatasetDict(train=ds["train"].shuffle(seed=SEED).select(range(min(n_train, len(ds["train"])))),
                           val=ds["validation"], test=ds["test"])
    except Exception as e:
        print("Không tải được UIT-VSFC → dùng dữ liệu mini. Lý do:", str(e)[:80])
    pos = ["Thầy giảng bài rất dễ hiểu và nhiệt tình.", "Môn học bổ ích, áp dụng được vào thực tế.",
           "Bài tập vừa sức, giúp em hiểu sâu kiến thức.", "Cô luôn hỗ trợ sinh viên ngoài giờ học.",
           "Phòng thí nghiệm hiện đại, thiết bị đầy đủ.", "Giáo trình rõ ràng, có nhiều ví dụ minh hoạ.",
           "Em rất thích cách thầy đặt câu hỏi gợi mở.", "Đề thi sát nội dung, đánh giá công bằng.",
           "Lớp học sôi nổi, mọi người tích cực thảo luận.", "Thầy phản hồi email rất nhanh.",
           "Slide đẹp, trình bày logic, dễ theo dõi.", "Thực hành trên Colab rất tiện, chạy được ngay."]
    neg = ["Thầy giảng quá nhanh, em không theo kịp.", "Bài tập quá nhiều so với số tín chỉ.",
           "Phòng học nóng, máy chiếu thường xuyên hỏng.", "Giáo trình cũ, nhiều nội dung lỗi thời.",
           "Đề thi quá khó, không liên quan bài giảng.", "Cô ít khi trả lời câu hỏi của sinh viên.",
           "Wifi yếu, không làm thực hành được.", "Lịch học thay đổi liên tục, rất bất tiện.",
           "Slide sơ sài, chỉ đọc lại sách.", "Điểm quá trình chấm không minh bạch.",
           "Máy tính phòng lab quá cũ, chạy rất chậm.", "Hướng dẫn đồ án không rõ ràng."]
    full = HFDataset.from_dict({"text": pos + neg, "label": [1] * len(pos) + [0] * len(neg)}).shuffle(seed=SEED)
    split = full.train_test_split(test_size=0.25, seed=SEED)
    return DatasetDict(train=split["train"], val=split["test"], test=split["test"])

vi_raw = load_vi_sentiment()
print(vi_raw)

In [ ]:
MODEL = "vinai/phobert-base-v2"          # thử: "uitnlp/visobert", "FacebookAI/xlm-roberta-base"
NEED_SEG = "phobert" in MODEL.lower()
vi_tok = AutoTokenizer.from_pretrained(MODEL)

def vi_preprocess(batch):
    texts = [normalize_vi(t) for t in batch["text"]]
    if NEED_SEG:
        texts = [word_segment(t) for t in texts]
    return vi_tok(texts, truncation=True, max_length=256)     # PhoBERT: tối đa 256 token

vi_tokenized = vi_raw.map(vi_preprocess, batched=True, remove_columns=["text"])
vi_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=2, id2label={0: "TIÊU_CỰC", 1: "TÍCH_CỰC"}, label2id={"TIÊU_CỰC": 0, "TÍCH_CỰC": 1})

vi_args = TrainingArguments(
    output_dir="vi_sentiment", num_train_epochs=4 if len(vi_tokenized["train"]) > 500 else 10,
    per_device_train_batch_size=16, per_device_eval_batch_size=64, learning_rate=3e-5,
    weight_decay=0.01, warmup_ratio=0.1, eval_strategy="epoch", save_strategy="epoch",
    logging_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="macro_f1",
    save_total_limit=1, fp16=torch.cuda.is_available(), report_to="none", seed=SEED)
vi_trainer = Trainer(model=vi_model, args=vi_args, train_dataset=vi_tokenized["train"],
                     eval_dataset=vi_tokenized["val"], processing_class=vi_tok,
                     data_collator=DataCollatorWithPadding(vi_tok), compute_metrics=compute_metrics)
vi_trainer.train()
print("Kết quả trên tập test:", vi_trainer.evaluate(vi_tokenized["test"]))

In [ ]:
# Phân tích lỗi trên các hiện tượng khó của tiếng Việt
hard_cases = {
    "Phủ định kép":  "Không phải là môn học dở.",
    "Phủ định":      "Thầy không hề nhiệt tình chút nào.",
    "Mỉa mai":       "Hay quá, học xong chẳng nhớ được gì.",
    "Tương phản":    "Nội dung hay nhưng thầy giảng quá nhanh.",
    "Teencode":      "mon nay ok lắm mn ơi, thầy dạy dễ hỉu",
    "Không dấu":     "thay giang rat de hieu va nhiet tinh",
    "Trộn Anh–Việt": "Deadline dày đặc, feedback thì chậm.",
}
vi_model.eval()
for kind, text in hard_cases.items():
    x = normalize_vi(text)
    x = word_segment(x) if NEED_SEG else x
    enc = vi_tok(x, return_tensors="pt", truncation=True, max_length=256).to(vi_model.device)
    with torch.inference_mode():
        p = vi_model(**enc).logits.softmax(-1)[0]
    print(f"{kind:14s} | {text:42s} → {vi_model.config.id2label[int(p.argmax())]:9s} ({p.max():.2f})")
print("\nThảo luận: lỗi nào do thiếu dữ liệu, lỗi nào do PhoBERT học trên văn bản chuẩn? So sánh với ViSoBERT.")

## 3.7 NER tiếng Việt: căn nhãn với fast tokenizer vs slow tokenizer

> **⏱ Ý chính.** Nhãn NER gán theo **từ**, nhưng mô hình nhìn thấy **subword**. Quy ước: nhãn ở subword đầu, subword sau nhận **−100** (bị loss bỏ qua).
> - **XLM-R** (fast): dùng `word_ids()` — gọn, chuẩn [40].
> - **PhoBERT** (slow, không có `word_ids()`): tự token hoá từng từ để căn [28, 34].

Với PhoBERT, đơn vị "từ" là **từ đã tách** (`Nguyễn_Văn_An`), nên nhãn BIO cũng phải theo từ đã tách — PhoNER_COVID19 cung cấp sẵn phiên bản word-level [34].

In [ ]:
labels_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
l2i = {l: i for i, l in enumerate(labels_list)}

# Mức âm tiết (cho XLM-R) và mức từ đã tách (cho PhoBERT) của cùng một câu
syll_words = ["TS.", "Hoàng", "Văn", "Quý", "giảng", "dạy", "tại", "Trường", "Đại", "học", "Thủy", "lợi", "ở", "Hà", "Nội", "."]
syll_tags  = ["O", "B-PER", "I-PER", "I-PER", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG", "I-ORG", "I-ORG", "O", "B-LOC", "I-LOC", "O"]
seg_words  = ["TS.", "Hoàng_Văn_Quý", "giảng_dạy", "tại", "Trường", "Đại_học", "Thủy_lợi", "ở", "Hà_Nội", "."]
seg_tags   = ["O", "B-PER", "O", "O", "B-ORG", "I-ORG", "I-ORG", "O", "B-LOC", "O"]

# (a) Fast tokenizer: word_ids()
def align_fast(words, tags, tok):
    enc = tok(words, is_split_into_words=True, truncation=True)
    labels, prev = [], None
    for wid in enc.word_ids():
        labels.append(-100 if wid is None or wid == prev else l2i[tags[wid]])
        prev = wid
    enc["labels"] = labels
    return enc

# (b) Slow tokenizer (PhoBERT): token hoá từng từ
def align_slow(words, tags, tok, max_len=256):
    ids, labels = [tok.cls_token_id], [-100]
    for w, t in zip(words, tags):
        pieces = tok.encode(w, add_special_tokens=False) or [tok.unk_token_id]
        ids += pieces
        labels += [l2i[t]] + [-100] * (len(pieces) - 1)
    ids, labels = ids[:max_len - 1] + [tok.sep_token_id], labels[:max_len - 1] + [-100]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

pho_tok = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
for name, enc, tok_ in [("XLM-R (fast, âm tiết)", align_fast(syll_words, syll_tags, xlmr_tok), xlmr_tok),
                        ("PhoBERT (slow, từ đã tách)", align_slow(seg_words, seg_tags, pho_tok), pho_tok)]:
    print(f"\n=== {name}: {len(enc['input_ids'])} token ===")
    for i, l in zip(enc["input_ids"], enc["labels"]):
        print(f"  {tok_.convert_ids_to_tokens(i):18s} {labels_list[l] if l != -100 else '-100'}")

In [ ]:
# Kiểm tra vòng huấn luyện NER chạy được (dữ liệu thật: PhoNER_COVID19 [34])
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification

ner_model = AutoModelForTokenClassification.from_pretrained(
    "vinai/phobert-base-v2", num_labels=len(labels_list),
    id2label=dict(enumerate(labels_list)), label2id=l2i).to(device)
toy = [(seg_words, seg_tags),
       (["Bà", "Trần_Thị_Bình", "sống", "ở", "Đà_Nẵng", "."], ["O", "B-PER", "O", "O", "B-LOC", "O"])]
batch = DataCollatorForTokenClassification(pho_tok)([align_slow(w, t, pho_tok) for w, t in toy]).to(device)
opt = AdamW(ner_model.parameters(), lr=5e-5)
ner_model.train()
for step in range(25):
    out = ner_model(**batch); opt.zero_grad(); out.loss.backward(); opt.step()
    if step % 8 == 0 or step == 24:
        print(f"step {step:2d}  loss = {out.loss.item():.4f}")

## 3.8 Embedding câu tiếng Việt và tìm kiếm ngữ nghĩa

> **⏱ Ý chính.** Vector `[CLS]` thô không phù hợp để đo tương đồng câu; cần mô hình huấn luyện **tương phản** (Sentence-BERT [35], E5 [36]). Đây là thành phần truy hồi của hệ RAG tiếng Việt.

In [ ]:
from sentence_transformers import SentenceTransformer

emb = SentenceTransformer("intfloat/multilingual-e5-small", device=device)
corpus = [
    "PhoBERT yêu cầu văn bản đầu vào đã được tách từ.",
    "ViSoBERT được huấn luyện trên dữ liệu mạng xã hội tiếng Việt.",
    "Hệ thống giám sát xói lở bờ sông dùng cảm biến IoT để cảnh báo sớm.",
    "Xâm nhập mặn ở Đồng bằng sông Cửu Long ngày càng nghiêm trọng.",
    "LoRA chỉ huấn luyện hai ma trận hạng thấp và giữ nguyên trọng số gốc.",
]
C = emb.encode(["passage: " + normalize_vi(c) for c in corpus], normalize_embeddings=True)
for q in ["Mô hình nào hợp với bình luận Facebook?", "Làm sao phát hiện sạt lở sớm?", "Fine-tune tiết kiệm bộ nhớ"]:
    s = C @ emb.encode(["query: " + normalize_vi(q)], normalize_embeddings=True)[0]
    best = int(s.argmax())
    print(f"{q:42s} → ({s[best]:.3f}) {corpus[best]}")

In [ ]:
# ============================================================
# 3.9 LLM HỘI THOẠI TIẾNG VIỆT: CHAT TEMPLATE + SINH THEO BATCH (padding trái)
# ============================================================
vi_questions = ["Giải thích ngắn gọn tokenizer là gì cho sinh viên năm nhất.",
                "Nêu hai lý do cần chuẩn hoá Unicode khi xử lý văn bản tiếng Việt."]
system = "Bạn là trợ giảng môn Xử lý ngôn ngữ tự nhiên. Trả lời bằng tiếng Việt, tối đa 3 câu."
texts = [chat_tok.apply_chat_template([{"role": "system", "content": system},
                                       {"role": "user", "content": q}],
                                      tokenize=False, add_generation_prompt=True) for q in vi_questions]
chat_tok.padding_side = "left"
batch = chat_tok(texts, return_tensors="pt", padding=True).to(device)
with torch.inference_mode():
    out = chat_model.generate(**batch, max_new_tokens=150, do_sample=False, repetition_penalty=1.05)
for q, o in zip(vi_questions, out):
    print("Hỏi:", q)
    print("Đáp:", chat_tok.decode(o[batch.input_ids.shape[1]:], skip_special_tokens=True), "\n")
print("Mô hình 0.5B chỉ để minh hoạ API; chất lượng tiếng Việt tốt hơn rõ rệt với bản 1.5B–7B.")

## 3.10 Checklist khi làm tiếng Việt với Hugging Face

| Bước | Việc cần làm | Lỗi thường gặp |
|---|---|---|
| Dữ liệu | `unicodedata.normalize("NFC")`, thống nhất kiểu dấu | Hai chuỗi "giống hệt" nhưng khác token |
| Tokenizer | Kiểm tra `is_fast`, fertility, `model_max_length` | PhoBERT > 256 token → lỗi index |
| PhoBERT | Tách từ (VnCoreNLP/underthesea) trước khi token hoá | Đưa văn bản thô → hiệu năng giảm |
| NER | Nhãn theo đúng đơn vị (âm tiết hay từ đã tách), subword phụ = −100 | Lệch nhãn khi chỉ đếm theo khoảng trắng |
| Tiền xử lý | **Không** bỏ stopword máy móc | Mất "không/chưa/chẳng" → sai cảm xúc |
| Đánh giá | Macro-F1, nhiều seed, báo cáo theo lớp | Chỉ báo accuracy trên dữ liệu lệch lớp |
| LLM | `apply_chat_template`, padding trái khi sinh batch | Tự ghép prompt sai định dạng |

# PHẦN 4 — BÀI TẬP

**Bài 1 (Trung bình) — Khám phá tokenizer.** Với 20 câu tiếng Việt thuộc lĩnh vực chuyên môn của em, so sánh 5 tokenizer ở mục 3.4 theo: fertility, tỉ lệ token `[UNK]`, độ dài tối đa. Vẽ biểu đồ và kết luận tokenizer nào phù hợp nhất. Gợi ý đọc: [26, 27].

**Bài 2 (Trung bình) — Vòng lặp PyTorch hoàn chỉnh.** Mở rộng mục 1.2: thêm gradient accumulation, mixed precision (`torch.autocast`), early stopping thủ công, lưu/tải checkpoint bằng `save_pretrained`. So sánh thời gian và kết quả với `Trainer`.

**Bài 3 (Khá) — PhoBERT vs ViSoBERT vs XLM-R trên UIT-VSFC [33].** Fine-tune 3 mô hình, mỗi mô hình 3 seed. Báo cáo macro-F1 (trung bình ± độ lệch chuẩn), thời gian/epoch, số tham số. Phân tích lỗi theo 7 nhóm hiện tượng ở mục 3.6.

**Bài 4 (Khá) — NER PhoNER_COVID19 [34].** Huấn luyện PhoBERT (bản đã tách từ) và XLM-R (bản âm tiết) dùng hai hàm căn nhãn ở mục 3.7. Đánh giá F1 mức thực thể bằng `evaluate.load("seqeval")`, báo cáo theo từng loại thực thể.

**Bài 5 (Nâng cao) — LoRA cho tiếng Việt.** Áp dụng mục 1.5 cho PhoBERT (`target_modules=["query","value"]`). Khảo sát $r \in \{4, 8, 16, 32\}$: macro-F1, số tham số huấn luyện, dung lượng adapter. So sánh với full fine-tuning [21, 22].

**Bài 6 (Nâng cao) — Khôi phục dấu tiếng Việt.** Tạo cặp (không dấu → có dấu) bằng `remove_accents`; fine-tune `VietAI/vit5-base` [31] hoặc `vinai/bartpho-syllable` [32] với `DataCollatorForSeq2Seq` và `Seq2SeqTrainer`. Đo độ chính xác theo âm tiết; phân tích lỗi trên các âm tiết đồng dạng khi bỏ dấu (ma/má/mà/mả/mã/mạ).

**Câu hỏi thảo luận**
1. Vì sao decoder-only cần padding trái khi sinh theo batch, còn encoder thì padding phải vẫn đúng?
2. Khi nào nên dùng `pipeline`, khi nào nên viết vòng lặp riêng?
3. Tách từ sai lan truyền sang NER như thế nào? Mô hình không cần tách từ (ViSoBERT, XLM-R) có tránh được vấn đề này không?
4. Hiện tượng "thuế token" [27] ảnh hưởng thế nào tới chi phí triển khai chatbot tiếng Việt dùng API LLM thương mại?

---

# TÀI LIỆU THAM KHẢO

### Thư viện và công cụ
1. Wolf, T., Debut, L., Sanh, V., et al. (2020). Transformers: State-of-the-art natural language processing. *Proceedings of EMNLP 2020: System Demonstrations*, 38–45.
2. Lhoest, Q., Villanova del Moral, A., Jernite, Y., et al. (2021). Datasets: A community library for natural language processing. *Proceedings of EMNLP 2021: System Demonstrations*, 175–184.

### Kiến trúc và mô hình nền tảng
3. Vaswani, A., Shazeer, N., Parmar, N., et al. (2017). Attention is all you need. *NeurIPS 2017*.
4. Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. *NAACL-HLT 2019*, 4171–4186.
5. Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter. *NeurIPS EMC² Workshop*. arXiv:1910.01108.
6. Liu, Y., Ott, M., Goyal, N., et al. (2019). RoBERTa: A robustly optimized BERT pretraining approach. arXiv:1907.11692.
7. Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). Language models are unsupervised multitask learners. *OpenAI Technical Report*.
8. Lewis, M., Liu, Y., Goyal, N., et al. (2020). BART: Denoising sequence-to-sequence pre-training for natural language generation, translation, and comprehension. *ACL 2020*, 7871–7880.
9. Raffel, C., Shazeer, N., Roberts, A., et al. (2020). Exploring the limits of transfer learning with a unified text-to-text transformer. *JMLR*, 21(140), 1–67.

### Mô hình, dữ liệu và phân tích dùng trong bài
10. Hartmann, J., Heitmann, M., Siebert, C., & Schamp, C. (2023). More than a feeling: Accuracy and application of sentiment analysis. *International Journal of Research in Marketing*, 40(1), 75–87. (Mô hình SiEBERT.)
11. Maas, A. L., Daly, R. E., Pham, P. T., Huang, D., Ng, A. Y., & Potts, C. (2011). Learning word vectors for sentiment analysis. *ACL-HLT 2011*, 142–150. (Bộ dữ liệu IMDB.)
12. Novikova, J., Dušek, O., & Rieser, V. (2017). The E2E dataset: New challenges for end-to-end generation. *SIGDIAL 2017*, 201–206.
13. Clark, K., Khandelwal, U., Levy, O., & Manning, C. D. (2019). What does BERT look at? An analysis of BERT's attention. *BlackboxNLP Workshop @ ACL 2019*, 276–286.

### Tokenization
14. Schuster, M., & Nakajima, K. (2012). Japanese and Korean voice search. *ICASSP 2012*, 5149–5152. (WordPiece.)
15. Sennrich, R., Haddow, B., & Birch, A. (2016). Neural machine translation of rare words with subword units. *ACL 2016*, 1715–1725. (BPE.)
16. Kudo, T., & Richardson, J. (2018). SentencePiece: A simple and language independent subword tokenizer and detokenizer for neural text processing. *EMNLP 2018: System Demonstrations*, 66–71.

### Huấn luyện, fine-tuning và sinh văn bản
17. Loshchilov, I., & Hutter, F. (2019). Decoupled weight decay regularization. *ICLR 2019*. (AdamW.)
18. Mosbach, M., Andriushchenko, M., & Klakow, D. (2021). On the stability of fine-tuning BERT: Misconceptions, explanations, and strong baselines. *ICLR 2021*.
19. Dodge, J., Ilharco, G., Schwartz, R., Farhadi, A., Hajishirzi, H., & Smith, N. (2020). Fine-tuning pretrained language models: Weight initializations, data orders, and early stopping. arXiv:2002.06305.
20. Holtzman, A., Buys, J., Du, L., Forbes, M., & Choi, Y. (2020). The curious case of neural text degeneration. *ICLR 2020*. (Top-p sampling.)
21. Hu, E. J., Shen, Y., Wallis, P., et al. (2022). LoRA: Low-rank adaptation of large language models. *ICLR 2022*.
22. Mangrulkar, S., Gugger, S., Debut, L., Belkada, Y., Paul, S., & Bossan, B. (2022). PEFT: State-of-the-art parameter-efficient fine-tuning methods. GitHub: huggingface/peft.
23. Dao, T., Fu, D. Y., Ermon, S., Rudra, A., & Ré, C. (2022). FlashAttention: Fast and memory-efficient exact attention with IO-awareness. *NeurIPS 2022*.
24. Micikevicius, P., Narang, S., Alben, J., et al. (2018). Mixed precision training. *ICLR 2018*.

### Đa ngữ và tokenizer đa ngữ
25. Conneau, A., Khandelwal, K., Goyal, N., et al. (2020). Unsupervised cross-lingual representation learning at scale. *ACL 2020*, 8440–8451. (XLM-R.)
26. Rust, P., Pfeiffer, J., Vulić, I., Ruder, S., & Gurevych, I. (2021). How good is your tokenizer? On the monolingual performance of multilingual language models. *ACL-IJCNLP 2021*, 3118–3135.
27. Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language model tokenizers introduce unfairness between languages. *NeurIPS 2023*.

### NLP tiếng Việt
28. Nguyen, D. Q., & Nguyen, A. T. (2020). PhoBERT: Pre-trained language models for Vietnamese. *Findings of EMNLP 2020*, 1037–1042.
29. Vu, T., Nguyen, D. Q., Nguyen, D. Q., Dras, M., & Johnson, M. (2018). VnCoreNLP: A Vietnamese natural language processing toolkit. *NAACL-HLT 2018: Demonstrations*, 56–60.
30. Nguyen, Q.-N., Phan, T. C., Nguyen, D.-V., & Nguyen, K. V. (2023). ViSoBERT: A pre-trained language model for Vietnamese social media text processing. *EMNLP 2023*.
31. Phan, L., Tran, H., Nguyen, H., & Trinh, T. H. (2022). ViT5: Pretrained text-to-text transformer for Vietnamese language generation. *NAACL 2022: Student Research Workshop*.
32. Tran, N. L., Le, D. M., & Nguyen, D. Q. (2022). BARTpho: Pre-trained sequence-to-sequence models for Vietnamese. *Interspeech 2022*.
33. Nguyen, K. V., Nguyen, V. D., Nguyen, P. X. V., Truong, T. T. H., & Nguyen, N. L.-T. (2018). UIT-VSFC: Vietnamese students' feedback corpus for sentiment analysis. *KSE 2018*.
34. Truong, T. H., Dao, M. H., & Nguyen, D. Q. (2021). COVID-19 named entity recognition for Vietnamese. *NAACL-HLT 2021*.

### Embedding và LLM
35. Reimers, N., & Gurevych, I. (2019). Sentence-BERT: Sentence embeddings using Siamese BERT-networks. *EMNLP-IJCNLP 2019*.
36. Wang, L., Yang, N., Huang, X., Yang, L., Majumder, R., & Wei, F. (2024). Multilingual E5 text embeddings: A technical report. arXiv:2402.05672.
37. Qwen Team. (2024). Qwen2.5 technical report. arXiv:2412.15115.

### Sách, khoá học, tài liệu kỹ thuật
38. Tunstall, L., von Werra, L., & Wolf, T. (2022). *Natural Language Processing with Transformers* (revised ed.). O'Reilly Media.
39. Stanford CS224N. *Hugging Face Transformers Tutorial* (tác giả gốc: Ben Newman; cập nhật Winter 2026: Minsik Oh). Khung ví dụ Phần 0–2 được phát triển từ tutorial này.
40. Hugging Face Documentation: Transformers (Auto Classes, Tokenizers, Trainer, Callbacks, Generation, Chat templating, Pipelines), Datasets, PEFT, LLM Course. https://huggingface.co/docs và https://huggingface.co/learn

---
*© Tài liệu giảng dạy học phần "Deep Learning trong NLP" — TS. Hoàng Văn Quý. Vui lòng không phân phối ngoài phạm vi lớp học khi chưa có sự đồng ý của giảng viên.*